# Session 2: Deployment and Scaling (Student Notebook)

In this session you will learn how to take LLM applications from prototype to production. We cover API design with FastAPI, caching strategies, monitoring and observability, model routing for cost optimization, and cost tracking — the operational skills that separate demos from deployed systems.

## Learning Objectives

By the end of this session, you will be able to:

1. **Design** a FastAPI service for LLM inference
2. **Implement** semantic caching to reduce API costs and latency
3. **Add** monitoring and structured logging for LLM applications
4. **Build** a model router that selects models based on complexity
5. **Track** token usage and costs across multiple models

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
import hashlib
import logging
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Designing an LLM API Service

Production LLM applications need a clean API layer. We design a service class that wraps the LLM with request validation, error handling, and structured responses — the same patterns you would use in a FastAPI endpoint.

In [ ]:
# Demo 1 - LLM API Service Design

@dataclass
class LLMRequest:
    """Validated request for the LLM service."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.7
    request_id: str = ""
    
    def __post_init__(self):
        if not self.request_id:
            self.request_id = hashlib.md5(f"{self.prompt}{time.time()}".encode()).hexdigest()[:8]
        if self.max_tokens < 1 or self.max_tokens > 4096:
            raise ValueError("max_tokens must be between 1 and 4096")
        if self.temperature < 0 or self.temperature > 2:
            raise ValueError("temperature must be between 0 and 2")

@dataclass
class LLMResponse:
    """Structured response from the LLM service."""
    content: str
    model: str
    request_id: str
    tokens_used: int
    latency_ms: float
    status: str = "success"
    error: Optional[str] = None

class LLMService:
    """Production LLM service with validation and error handling."""
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def invoke(self, request: LLMRequest) -> LLMResponse:
        start = time.time()
        try:
            response = self.client.chat.completions.create(
                model=request.model,
                messages=[{"role": "user", "content": request.prompt}],
                max_tokens=request.max_tokens,
                temperature=request.temperature
            )
            latency = (time.time() - start) * 1000
            return LLMResponse(
                content=response.choices[0].message.content,
                model=request.model,
                request_id=request.request_id,
                tokens_used=response.usage.total_tokens,
                latency_ms=round(latency, 2)
            )
        except Exception as e:
            latency = (time.time() - start) * 1000
            return LLMResponse(
                content="", model=request.model,
                request_id=request.request_id,
                tokens_used=0, latency_ms=round(latency, 2),
                status="error", error=str(e)
            )

# Test the service
service = LLMService()
req = LLMRequest(prompt="What is RAG in 2 sentences?", max_tokens=100, temperature=0)
resp = service.invoke(req)

print(f"Request ID: {resp.request_id}")
print(f"Status: {resp.status}")
print(f"Model: {resp.model}")
print(f"Tokens: {resp.tokens_used}")
print(f"Latency: {resp.latency_ms}ms")
print(f"Response: {resp.content}")

### Demo 2: Semantic Caching for LLM Responses

LLM calls are expensive and slow. Semantic caching stores previous responses and returns them for similar (not just identical) queries — reducing costs by 30-70% in typical production workloads.

In [ ]:
# Demo 2 - Semantic Caching

class SemanticCache:
    """Cache that matches by semantic similarity, not exact string match."""
    
    def __init__(self, similarity_threshold=0.92):
        self.client = openai.OpenAI()
        self.cache = []  # List of (embedding, query, response)
        self.threshold = similarity_threshold
        self.hits = 0
        self.misses = 0
    
    def _embed(self, text):
        return self.client.embeddings.create(
            model="text-embedding-3-small", input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get(self, query):
        """Check cache for a semantically similar query."""
        query_emb = self._embed(query)
        best_sim, best_response = 0, None
        
        for cached_emb, cached_query, cached_response in self.cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = (cached_query, cached_response)
        
        if best_sim >= self.threshold:
            self.hits += 1
            return {"hit": True, "similarity": best_sim,
                    "cached_query": best_response[0], "response": best_response[1]}
        self.misses += 1
        return {"hit": False, "similarity": best_sim}
    
    def put(self, query, response):
        """Store a query-response pair in cache."""
        emb = self._embed(query)
        self.cache.append((emb, query, response))

# Test
cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# First query — cache miss
q1 = "What is retrieval augmented generation?"
result = cache.get(q1)
print(f"Query: {q1}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")

# Call LLM and cache the response
answer = llm.invoke([HumanMessage(content=q1)]).content
cache.put(q1, answer)
print(f"Cached response: {answer[:100]}...")

# Similar query — should hit cache
q2 = "Explain RAG to me"
result = cache.get(q2)
print(f"\nQuery: {q2}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
if result['hit']:
    print(f"Matched to: {result['cached_query']}")

# Different query — should miss
q3 = "How does gradient descent work?"
result = cache.get(q3)
print(f"\nQuery: {q3}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
print(f"\nStats: {cache.hits} hits, {cache.misses} misses")

### Demo 3: Monitoring and Structured Logging

In production, you need to know what your LLM is doing. Structured logging captures every request, response, token count, latency, and error in a queryable format — essential for debugging and optimization.

In [ ]:
# Demo 3 - Monitoring and Structured Logging

class LLMMonitor:
    """Monitors LLM usage with structured logging and metrics."""
    
    def __init__(self):
        self.logs = []
        self.metrics = defaultdict(list)
    
    def log_request(self, request_id, model, prompt, response, tokens, latency_ms, status="success"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id,
            "model": model,
            "prompt_preview": prompt[:100],
            "response_preview": response[:100] if response else "",
            "tokens": tokens,
            "latency_ms": latency_ms,
            "status": status
        }
        self.logs.append(entry)
        self.metrics["latency"].append(latency_ms)
        self.metrics["tokens"].append(tokens)
        self.metrics["models"].append(model)
        return entry
    
    def get_summary(self):
        if not self.logs:
            return "No requests logged."
        latencies = self.metrics["latency"]
        tokens = self.metrics["tokens"]
        models = self.metrics["models"]
        return {
            "total_requests": len(self.logs),
            "avg_latency_ms": round(np.mean(latencies), 2),
            "p95_latency_ms": round(np.percentile(latencies, 95), 2),
            "total_tokens": sum(tokens),
            "avg_tokens": round(np.mean(tokens), 2),
            "model_distribution": {m: models.count(m) for m in set(models)},
            "error_rate": sum(1 for l in self.logs if l["status"] != "success") / len(self.logs)
        }

# Test
monitor = LLMMonitor()
client = openai.OpenAI()

prompts = [
    "What is RAG?",
    "Explain vector databases.",
    "How does chunking work?",
    "What is LangChain?",
    "Compare FAISS and ChromaDB."
]

for prompt in prompts:
    start = time.time()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80
    )
    latency = (time.time() - start) * 1000
    req_id = hashlib.md5(prompt.encode()).hexdigest()[:8]
    
    entry = monitor.log_request(
        request_id=req_id, model="gpt-4o-mini",
        prompt=prompt, response=resp.choices[0].message.content,
        tokens=resp.usage.total_tokens, latency_ms=round(latency, 2)
    )
    print(f"[{entry['request_id']}] {entry['prompt_preview'][:40]:40s} | {entry['tokens']}tok | {entry['latency_ms']}ms")

print("\n--- Summary ---")
summary = monitor.get_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

### Demo 4: Model Routing for Cost Optimization

Not every query needs GPT-4. A model router classifies query complexity and sends simple queries to cheaper/faster models, reserving expensive models for complex tasks — cutting costs by 40-60% with minimal quality loss.

In [ ]:
# Demo 4 - Model Routing

class ModelRouter:
    """Routes queries to appropriate models based on complexity."""
    
    MODEL_TIERS = {
        "simple": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "medium": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "complex": {"model": "gpt-4o", "cost_per_1k": 0.005}
    }
    
    def __init__(self):
        self.classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    def classify_complexity(self, query):
        """Classify query complexity to determine model tier."""
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this query's complexity. Return ONLY one word:
- simple: factual lookups, definitions, short answers
- medium: explanations, comparisons, moderate reasoning
- complex: multi-step reasoning, code generation, analysis"""),
            HumanMessage(content=query)
        ])
        complexity = response.content.strip().lower()
        if complexity not in self.MODEL_TIERS:
            complexity = "medium"
        return complexity
    
    def route(self, query):
        """Route query to the appropriate model."""
        complexity = self.classify_complexity(query)
        tier = self.MODEL_TIERS[complexity]
        return {"complexity": complexity, "model": tier["model"], "cost_per_1k": tier["cost_per_1k"]}

# Test
router = ModelRouter()
test_queries = [
    "What does RAG stand for?",
    "Compare vector databases: ChromaDB vs FAISS vs Pinecone.",
    "Write a Python function that implements a RAG pipeline with reranking, caching, and evaluation metrics.",
    "What is an embedding?",
    "Design a multi-agent system for automated code review with test generation."
]

for q in test_queries:
    result = router.route(q)
    print(f"[{result['complexity']:7s}] -> {result['model']:12s} (${result['cost_per_1k']}/1k) | {q[:60]}")

### Demo 5: Token Usage and Cost Tracking

Tracking costs is critical for budget management. This demo builds a cost tracker that monitors token usage per model, calculates running costs, and provides spending alerts.

In [ ]:
# Demo 5 - Token Usage and Cost Tracking

class CostTracker:
    """Tracks token usage and costs across models."""
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "text-embedding-3-small": {"input": 0.00002, "output": 0}
    }
    
    def __init__(self, budget_limit=1.0):
        self.budget_limit = budget_limit
        self.records = []
    
    def record(self, model, input_tokens, output_tokens, request_id=""):
        pricing = self.PRICING.get(model, {"input": 0.001, "output": 0.002})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        entry = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost,
            "request_id": request_id
        }
        self.records.append(entry)
        total = sum(r["cost"] for r in self.records)
        if total > self.budget_limit * 0.8:
            print(f"  WARNING: Budget at {total/self.budget_limit*100:.1f}% (${total:.6f}/${self.budget_limit})")
        return entry
    
    def get_report(self):
        total_cost = sum(r["cost"] for r in self.records)
        by_model = defaultdict(lambda: {"requests": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0})
        for r in self.records:
            by_model[r["model"]]["requests"] += 1
            by_model[r["model"]]["input_tokens"] += r["input_tokens"]
            by_model[r["model"]]["output_tokens"] += r["output_tokens"]
            by_model[r["model"]]["cost"] += r["cost"]
        return {"total_cost": total_cost, "total_requests": len(self.records),
                "budget_remaining": self.budget_limit - total_cost, "by_model": dict(by_model)}

# Test
tracker = CostTracker(budget_limit=0.05)
client = openai.OpenAI()

queries = [
    ("gpt-4o-mini", "What is RAG?"),
    ("gpt-4o-mini", "List three vector databases."),
    ("gpt-4o", "Design a production RAG architecture with caching and monitoring."),
    ("gpt-4o-mini", "What is cosine similarity?"),
]

for model, prompt in queries:
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], max_tokens=100
    )
    entry = tracker.record(model, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print(f"[{model:12s}] {prompt[:40]:40s} | ${entry['cost']:.6f}")

print("\n--- Cost Report ---")
report = tracker.get_report()
print(f"Total cost: ${report['total_cost']:.6f}")
print(f"Budget remaining: ${report['budget_remaining']:.6f}")
for model, stats in report['by_model'].items():
    print(f"  {model}: {stats['requests']} reqs, {stats['input_tokens']+stats['output_tokens']} tokens, ${stats['cost']:.6f}")

---
## Tasks (Your Turn!)

### Task 1: Build a Rate-Limited LLM Service

Create an LLM service wrapper that enforces rate limits — both requests-per-minute and tokens-per-minute — to prevent runaway costs and API throttling.

In [ ]:
# Task 1 - Build a Rate-Limited LLM Service

# TODO: Build a RateLimitedLLM class that:
# 1. Tracks requests and tokens in a sliding window (last 60 seconds)
# 2. Blocks requests that would exceed the rate limit
# 3. Returns a helpful error message when rate limited
#
# Hint: Store timestamps of recent requests and their token counts
# Hint: Before each request, prune entries older than 60 seconds
# Hint: Check both RPM and TPM limits before making the API call

class RateLimitedLLM:
    def __init__(self, rpm_limit=10, tpm_limit=5000):
        # YOUR CODE HERE
        pass

    def invoke(self, prompt, max_tokens=200):
        # YOUR CODE HERE
        pass


# Test
# rl_llm = RateLimitedLLM(rpm_limit=3, tpm_limit=1000)
# for i in range(5):
#     result = rl_llm.invoke(f"Question {i}: What is RAG?")
#     print(f"Request {i}: {result['status']}")

### Task 2: Implement a Response Streaming Simulator

Build a streaming response handler that processes tokens as they arrive, tracks partial results, and provides real-time latency metrics (time-to-first-token, tokens-per-second).

In [ ]:
# Task 2 - Implement a Response Streaming Simulator

# TODO: Build a StreamingHandler class that:
# 1. Uses the OpenAI streaming API (stream=True)
# 2. Tracks time-to-first-token (TTFT) and tokens-per-second
# 3. Collects the full response while reporting progress
#
# Hint: Use client.chat.completions.create(..., stream=True)
# Hint: Iterate over chunks: for chunk in response: ...
# Hint: Track time between first call and first token for TTFT

class StreamingHandler:
    def __init__(self):
        # YOUR CODE HERE
        pass

    def stream(self, prompt, model="gpt-4o-mini", max_tokens=200):
        # YOUR CODE HERE
        pass


# Test
# handler = StreamingHandler()
# result = handler.stream("Explain RAG in 3 sentences.")
# print(f"TTFT: {result['ttft_ms']:.0f}ms")
# print(f"Tokens/sec: {result['tokens_per_sec']:.1f}")
# print(f"Full response: {result['content']}")

### Task 3: Build a Multi-Tier Caching System

Build a caching system with two tiers: exact-match (fast, hash-based) and semantic-match (slower, embedding-based). Queries check exact cache first, then semantic cache, then call the LLM.

In [ ]:
# Task 3 - Build a Multi-Tier Caching System

# TODO: Build a TieredCache class that:
# 1. Has an exact-match tier (dict-based, O(1) lookup)
# 2. Has a semantic-match tier (embedding similarity)
# 3. Falls through: exact -> semantic -> LLM call
# 4. Reports which tier served the response
#
# Hint: Use a dict for exact match (key = prompt hash)
# Hint: Use embedding comparison for semantic match
# Hint: Track hit rates for each tier

class TieredCache:
    def __init__(self, semantic_threshold=0.92):
        # YOUR CODE HERE
        pass

    def query(self, prompt):
        # YOUR CODE HERE
        pass


# Test
# cache = TieredCache()
# print(cache.query("What is RAG?"))        # LLM call
# print(cache.query("What is RAG?"))        # Exact match
# print(cache.query("Explain RAG to me"))   # Semantic match
# print(cache.query("How does gravity work?"))  # LLM call

### Task 4: Create a Full Production LLM Gateway

Combine everything into a production gateway: model routing, caching, rate limiting, monitoring, and cost tracking — all in one unified service.

In [ ]:
# Task 4 - Create a Full Production LLM Gateway

# TODO: Build an LLMGateway class that combines:
# 1. Model routing (simple -> mini, complex -> 4o)
# 2. Caching (check cache before calling LLM)
# 3. Rate limiting (enforce RPM/TPM limits)
# 4. Monitoring (log every request)
# 5. Cost tracking (track spend per model)
#
# Hint: Compose the components from previous demos/tasks
# Hint: Pipeline: route -> check cache -> check rate limit -> call LLM -> cache response -> log
# Hint: Return a rich response with answer, model used, cache status, cost

class LLMGateway:
    def __init__(self):
        # YOUR CODE HERE
        pass

    def query(self, prompt):
        # YOUR CODE HERE
        pass

    def get_dashboard(self):
        # YOUR CODE HERE
        pass


# Test
# gateway = LLMGateway()
# for q in ["What is RAG?", "What is RAG?", "Design a multi-agent system."]:
#     result = gateway.query(q)
#     print(f"{q[:40]:40s} | model={result['model']} cache={result['cached']} cost=${result['cost']:.6f}")
# print(gateway.get_dashboard())

---
## Summary

In this session you learned the operational skills for production LLM deployment:

1. **API Design** -- How to structure LLM services with validation and error handling.
2. **Semantic Caching** -- How to reduce costs by caching similar (not just identical) queries.
3. **Monitoring** -- How to track latency, tokens, errors, and model usage.
4. **Model Routing** -- How to route queries to cheaper models when possible.
5. **Cost Tracking** -- How to monitor spending and set budget alerts.

**Up Next:** After lunch, you will choose your capstone track and build a complete system.